# Predicting code-switches 



**Plan**

2. Build a **causal** (forward-only) LSTM whose hidden state at step $t$
   summarizes only past context, and train it to predict $\text{switch}_{t+1}$.
3. Build a **classical ML predictor** (logistic regression, decision tree,
   random forest, gradient boosting) over hand-engineered causal features
   — the Solorio & Liu 2008 style baseline that set the pre-deep-learning
   standard for this task.
4. Compare both against three reference points: a trivial "always-stay"
   baseline, a classical **dual unigram LM** (which actually peeks at
   $w_{t+1}$'s form and therefore tells us how much the current word
   contributes), and Method B from the previous notebook (full BiLSTM
   detector, not a predictor at all).
5. Read the comparison table and be honest about what each number means.


## 1. Detection vs. prediction — the information each model sees

Think of the same token stream at position $t$:

| Approach | Inputs visible at step $t$ | Target at step $t$ |
|---|---|---|
| **Detection** (BiLSTM) | $w_1,\dots,w_{T}$ — past and future, including $w_t$ itself | Language of $w_t$ |
| **Prediction** (causal LSTM) | $w_1,\dots,w_t$ — past only, **not** $w_{t+1}$ | Whether $w_{t+1}$ will be a switch |

Bidirectionality is the leak: the backward pass of a BiLSTM at position
$t$ is computed from the *whole suffix*, so the hidden state encodes
information that a true forecaster couldn't know yet. Dropping the
backward direction restores causality.

A recurrent neural network is actually the most natural architecture for
prediction because it's causal by construction:

```
h_0 → h_1 → h_2 → ... → h_t
       ↑     ↑            ↑
      w_1   w_2          w_t
```

$h_t$ is a summary of **only** words $1\dots t$, so anything we compute
from $h_t$ strictly respects the past-only constraint.

### Aligning inputs to targets

The small but crucial trick: use $h_t$ to predict what happens at
$t{+}1$. Concretely, if we feed the sentence $w_1 \dots w_T$ through a
forward LSTM and read off outputs $o_1 \dots o_T$, we shift by one:

* $o_t$ predicts $\text{switch}_{t+1}$.
* Loss is computed on positions $2 \dots T$.
* The last output $o_T$ has no target — there is no $w_{T+1}$.

This is the same setup language models use for next-token prediction.

## 2. Setup

We reuse the dialogue-level 80/10/10 split.

In [1]:
import os, time, random, json, math
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score,
    average_precision_score,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CORPUS = Path("../Data/CS Corpus Prediction/BangorCorpus.txt")
df = pd.read_csv(CORPUS, sep="\t")


In [2]:
TOP_LANGS = ("eng", "spa")
def tokenize(utt, syn):
    if not isinstance(utt, str) or not isinstance(syn, str): return None
    toks, tags = utt.split(","), syn.split(".")
    if len(toks) != len(tags): return None
    out = []
    for tok, pos in zip(toks, tags):
        if "." not in tok: return None
        w, l = tok.rsplit(".", 1)
        out.append((w.strip(), l.strip(), pos.strip()))
    return out
def ok(triples, allowed=("eng","spa","amb")):
    return all(l in allowed for _, l, _ in triples)

rows = []
dialogue_to_id = {sf: i for i, sf in enumerate(sorted(df["Soundfile"].unique()))}
sent_cnt = defaultdict(int)
for _, r in df.iterrows():
    t = tokenize(r["Utterance"], r["Syntax"])
    if t is None or not ok(t): continue
    did = dialogue_to_id[r["Soundfile"]]; sent_cnt[did] += 1; sid = sent_cnt[did]
    for wi, (w, l, p) in enumerate(t, 1):
        rows.append({"dialogue_id": did, "sentence_id": sid,
                     "word_index": wi, "word": w, "word_lang": l, "pos_tag": p})
tidy = pd.DataFrame(rows)

data = tidy[tidy["word_lang"].isin(TOP_LANGS)].copy().reset_index(drop=True)
data["y"] = (data["word_lang"] == "spa").astype(int)

rng = np.random.default_rng(SEED)
dials = np.array(sorted(data["dialogue_id"].unique())); rng.shuffle(dials)
n = len(dials); n_tr, n_va = int(0.8*n), int(0.1*n)
tr_d = set(dials[:n_tr]); va_d = set(dials[n_tr:n_tr+n_va])
data["split"] = data["dialogue_id"].map(
    lambda d: "train" if d in tr_d else ("val" if d in va_d else "test"))
data = data.sort_values(["dialogue_id","sentence_id","word_index"]).reset_index(drop=True)

grp = data.groupby(["dialogue_id","sentence_id"], sort=False)
data["prev_y"] = grp["y"].shift(1)
data["switch_true"] = (data["prev_y"].notna() &
                       (data["y"] != data["prev_y"])).astype("float")

print("Split sizes:", data.groupby("split").size().to_dict())
print("Overall within-sentence switch rate:",
      round(data.loc[data["prev_y"].notna(), "switch_true"].mean(), 4))


Split sizes: {'test': 37027, 'train': 182997, 'val': 28134}
Overall within-sentence switch rate: 0.0178


About **1.8% of within-sentence tokens** are switches. Keep that number
handy — it's the reason an "always-stay" predictor can score 98%
accuracy while being useless. It's also the reason `pos_weight` will do
a lot of work during training.

## 3. Vocabularies and sentence tensors

Same word/POS/char vocabularies as before, fit on training data only.

In [3]:
PAD, UNK = "<pad>", "<unk>"
tr_words = data[data["split"]=="train"]

wc = Counter(tr_words["word"].tolist())
vocab = [PAD, UNK] + [w for w, c in wc.most_common() if c >= 2]
word2id = {w:i for i,w in enumerate(vocab)}
pos_vocab = [PAD] + sorted(tr_words["pos_tag"].unique().tolist())
pos2id = {p:i for i,p in enumerate(pos_vocab)}

char_counter = Counter()
for w in tr_words["word"]:
    for ch in w.lower():
        char_counter[ch] += 1
chars = [PAD, UNK] + [c for c, cnt in char_counter.most_common() if cnt >= 5]
char2id = {c:i for i,c in enumerate(chars)}

print(f"Word vocab {len(word2id):,}  |  POS vocab {len(pos2id)}  |  Char vocab {len(char2id)}")

enc_w = lambda w: word2id.get(w, word2id[UNK])
enc_p = lambda p: pos2id.get(p, 0)
MAX_CHAR = 20
def enc_chars(w, max_c=MAX_CHAR):
    w = w.lower()[:max_c]
    ids = [char2id.get(c, char2id[UNK]) for c in w]
    return ids + [0] * (max_c - len(ids))

def build_sents(df_):
    out = []
    for (d,s), g in df_.groupby(["dialogue_id","sentence_id"], sort=False):
        g = g.sort_values("word_index")
        y = g["y"].astype(int).tolist()
        out.append({
            "did": int(d), "sid": int(s),
            "words_id": [enc_w(w) for w in g["word"]],
            "chars":    [enc_chars(w) for w in g["word"]],
            "pos":      [enc_p(p) for p in g["pos_tag"]],
            "y":        y,
            "switch":   [0] + [int(y[i] != y[i-1]) for i in range(1, len(y))],
        })
    return out

tr_sents = build_sents(data[data["split"]=="train"])
va_sents = build_sents(data[data["split"]=="val"])
te_sents = build_sents(data[data["split"]=="test"])
print(f"Sentences: train {len(tr_sents)}  val {len(va_sents)}  test {len(te_sents)}")


Word vocab 5,599  |  POS vocab 48  |  Char vocab 38
Sentences: train 32300  val 4628  test 6399


In [4]:
class SentDS(Dataset):
    def __init__(self, sents): self.sents = sents
    def __len__(self): return len(self.sents)
    def __getitem__(self, i):
        s = self.sents[i]
        return (torch.tensor(s["words_id"], dtype=torch.long),
                torch.tensor(s["pos"],      dtype=torch.long),
                torch.tensor(s["chars"],    dtype=torch.long),
                torch.tensor(s["switch"],   dtype=torch.float))

def collate(batch):
    W = pad_sequence([b[0] for b in batch], batch_first=True, padding_value=0)
    P = pad_sequence([b[1] for b in batch], batch_first=True, padding_value=0)
    Lmax = max(b[2].shape[0] for b in batch)
    C = torch.zeros(len(batch), Lmax, MAX_CHAR, dtype=torch.long)
    for i, b in enumerate(batch):
        C[i, :b[2].shape[0]] = b[2]
    S = pad_sequence([b[3] for b in batch], batch_first=True, padding_value=0.0)
    M = (W != 0).float()
    return W, P, C, S, M

BATCH = 32
tr_loader = DataLoader(SentDS(tr_sents), BATCH, shuffle=True,  collate_fn=collate)
va_loader = DataLoader(SentDS(va_sents), BATCH, shuffle=False, collate_fn=collate)
te_loader = DataLoader(SentDS(te_sents), BATCH, shuffle=False, collate_fn=collate)


## 4. Method D — the forward only LSTM switch predictor


`bidirectional=False` 
 We **shift** the target: `logits[:, t]` (read off from $h_t$) is
   trained against `switch[:, t+1]`. The loss skips the last position of
   each sentence because there is no $w_{T+1}$ to predict.


In [5]:
class CharCNN(nn.Module):
    def __init__(self, n_chars, char_dim=16, out_dim=32,
                 kernel_sizes=(2,3,4), filters=16):
        super().__init__()
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, filters, k, padding=k//2) for k in kernel_sizes
        ])
        self.proj = nn.Linear(filters * len(kernel_sizes), out_dim)
    def forward(self, char_ids):
        B, L, C = char_ids.shape
        x = self.char_emb(char_ids).view(B*L, C, -1).transpose(1, 2)
        outs = [torch.relu(conv(x)).max(dim=-1).values for conv in self.convs]
        return self.proj(torch.cat(outs, dim=-1)).view(B, L, -1)

class CausalPredictor(nn.Module):
    def __init__(self, n_words, n_pos, n_chars,
                 wd=64, pd_=16, char_out=32,
                 hidden=128, dropout=0.3):
        super().__init__()
        self.we = nn.Embedding(n_words, wd, padding_idx=0)
        self.pe = nn.Embedding(n_pos,   pd_, padding_idx=0)
        self.char = CharCNN(n_chars, out_dim=char_out)
        self.lstm = nn.LSTM(wd + pd_ + char_out, hidden,
                            batch_first=True, bidirectional=False)  # <- causal
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)
    def forward(self, w, p, c):
        x = torch.cat([self.we(w), self.pe(p), self.char(c)], dim=-1)
        h, _ = self.lstm(x)                     # (B, T, H)
        return self.head(self.drop(h)).squeeze(-1)


### Training loop with the shift

The key line is `logits[:, :-1]` against `S[:, 1:]`. We also do
best-val checkpointing because the causal task is harder and training
starts to overfit after ~2 epochs.

In [6]:
sw_train = [t for s in tr_sents for t in s["switch"][1:]]
train_switch_rate = float(np.mean(sw_train))
pw = (1 - train_switch_rate) / max(train_switch_rate, 1e-6)
print(f"Train switch rate: {train_switch_rate:.4f}  ->  pos_weight {pw:.1f}")

def run_epoch(model, loader, crit, opt=None):
    train_mode = opt is not None
    model.train(train_mode)
    tot_loss, tot_tok = 0.0, 0.0
    ps, ys = [], []
    for W, P, C, S, M in loader:
        logits = model(W, P, C)
        lg  = logits[:, :-1]            # predictions for positions 2..T
        tgt = S[:, 1:]
        msk = M[:, 1:]
        loss_tok = crit(lg, tgt) * msk
        loss = loss_tok.sum() / msk.sum().clamp(min=1)
        if train_mode:
            opt.zero_grad(); loss.backward(); opt.step()
        tot_loss += loss.item() * msk.sum().item()
        tot_tok  += msk.sum().item()
        with torch.no_grad():
            prob = torch.sigmoid(lg).numpy()
            tg   = tgt.numpy(); mm = msk.numpy()
            for b in range(prob.shape[0]):
                L = int(mm[b].sum())
                ps.append(prob[b, :L]); ys.append(tg[b, :L])
    return tot_loss/max(tot_tok,1), np.concatenate(ps), np.concatenate(ys)

torch.manual_seed(SEED)
model_D = CausalPredictor(len(word2id), len(pos2id), len(char2id))
print(f"Params: {sum(p.numel() for p in model_D.parameters()):,}")

crit = nn.BCEWithLogitsLoss(reduction="none",
                            pos_weight=torch.tensor([pw]))
opt  = torch.optim.Adam(model_D.parameters(), lr=1e-3)

best_auc, best_state = -1.0, None
for ep in range(1, 5):
    tl, tp, ty = run_epoch(model_D, tr_loader, crit, opt)
    vl, vp, vy = run_epoch(model_D, va_loader, crit)
    ta, va = roc_auc_score(ty, tp), roc_auc_score(vy, vp)
    print(f"  ep {ep}  train loss {tl:.4f}  auc {ta:.3f} | "
          f"val loss {vl:.4f}  auc {va:.3f}")
    if va > best_auc:
        best_auc = va
        best_state = {k: v.clone() for k, v in model_D.state_dict().items()}

model_D.load_state_dict(best_state)
print(f"Best val AUC = {best_auc:.3f}")


Train switch rate: 0.0179  ->  pos_weight 54.9
Params: 487,665
  ep 1  train loss 1.1711  auc 0.745 | val loss 1.0460  auc 0.797
  ep 2  train loss 1.0584  auc 0.804 | val loss 1.0112  auc 0.802
  ep 3  train loss 0.9713  auc 0.840 | val loss 1.0851  auc 0.774
  ep 4  train loss 0.8790  auc 0.872 | val loss 1.1563  auc 0.761
Best val AUC = 0.802


Train AUC keeps climbing while val AUC peaks at epoch 2 which is overfitting.  Note also that val AUC ≈ 0.80 is far below the ~0.99 we saw
Need to check later:
Most common causes:

Dataset is small or low-diversity: model sees nearly the same patterns repeatedly and memorizes fast.
Model is too expressive: too many parameters for the amount of data.
Train/val mismatch: validation distribution differs from training (different speakers, domains, preprocessing, label policy).
Label noise: training loss keeps dropping, but val stops improving because labels are inconsistent.
Weak regularization: little/no dropout, weight decay, data augmentation, or early stopping.
Validation leakage issues: occasionally the split is not stratified/grouped correctly (or the reverse: train and val are too different due to bad split).

## 5. Evaluation — test split

We score each test token $r$ with `model_D`'s output at the *previous*
position, i.e. $P(\text{switch at } r \mid w_1 \dots w_{r-1})$. Then we
sweep the threshold on the validation split to pick the best-F1
operating point before freezing it on test.

In [7]:
def predict_for(sents, model):
    prob_into_row = np.full(len(data), np.nan)
    model.eval()
    with torch.no_grad():
        for s in sents:
            W = torch.tensor([s["words_id"]], dtype=torch.long)
            P = torch.tensor([s["pos"]],      dtype=torch.long)
            C = torch.tensor([s["chars"]],    dtype=torch.long)
            logits = model(W, P, C).numpy()[0]
            T = len(s["words_id"])
            mask = ((data["dialogue_id"]==s["did"])
                    & (data["sentence_id"]==s["sid"]))
            idxs = sorted(data.index[mask].tolist(),
                          key=lambda i: data.at[i, "word_index"])
            # logits[t-1] (hidden state at t-1) predicts whether position t is a switch
            for t in range(1, T):
                prob_into_row[idxs[t]] = 1/(1+np.exp(-logits[t-1]))
    return prob_into_row

probs_D_test = predict_for(te_sents, model_D)
probs_D_val  = predict_for(va_sents, model_D)

def eval_switch(probs, thr, split="test"):
    m = ((data["split"]==split) & data["prev_y"].notna()
         & (~np.isnan(probs)))
    yt = data.loc[m, "switch_true"].astype(int).values
    sc = probs[m.values]
    yp = (sc >= thr).astype(int)
    P, R, F, _ = precision_recall_fscore_support(yt, yp, average="binary",
                                                 zero_division=0)
    return {"n": int(m.sum()), "thr": float(thr),
            "P": float(P), "R": float(R), "F1": float(F),
            "AUC": float(roc_auc_score(yt, sc)),
            "AP":  float(average_precision_score(yt, sc))}

def tune_threshold(probs):
    m = ((data["split"]=="val") & data["prev_y"].notna()
         & (~np.isnan(probs)))
    yv = data.loc[m, "switch_true"].astype(int).values
    sv = probs[m.values]
    best_thr, best_F = 0.5, 0.0
    for thr in np.linspace(0.05, 0.99, 95):
        yp = (sv >= thr).astype(int)
        _, _, F, _ = precision_recall_fscore_support(yv, yp, average="binary",
                                                     zero_division=0)
        if F > best_F: best_F, best_thr = F, thr
    return best_thr, best_F

thr_D, valF_D = tune_threshold(probs_D_val)
print(f"Best val F1 = {valF_D:.3f} at threshold = {thr_D:.2f}")

res_D_05   = eval_switch(probs_D_test, 0.5)
res_D_tune = eval_switch(probs_D_test, thr_D)
print(f"Method D @ thr=0.50 :  P={res_D_05['P']:.3f}  R={res_D_05['R']:.3f}  "
      f"F1={res_D_05['F1']:.3f}  AUC={res_D_05['AUC']:.3f}")
print(f"Method D @ thr={thr_D:.2f} :  P={res_D_tune['P']:.3f}  R={res_D_tune['R']:.3f}  "
      f"F1={res_D_tune['F1']:.3f}  AUC={res_D_tune['AUC']:.3f}")


Best val F1 = 0.151 at threshold = 0.82
Method D @ thr=0.50 :  P=0.038  R=0.683  F1=0.072  AUC=0.751
Method D @ thr=0.82 :  P=0.077  R=0.263  F1=0.119  AUC=0.751


## 6. Baselines

### 6a. Always-stay

Predicts "not a switch" for every position. Scores 98%+ accuracy
(because the base rate of switches is ~1.8%) but zero recall on the
class that matters.

### 6b. Unigram dual-LM (looks at $w_{t+1}$ — a detection upper bound)

Train a word-frequency table per language on the training split and
classify each test word by its most likely language:

$$\hat{\ell}(w_{t+1}) = \arg\max_{\ell \in \{\text{eng,spa}\}} P_\ell(w_{t+1})$$

Predict a switch iff $\hat{\ell}(w_{t+1}) \ne \ell(w_t)$.

**Important caveat**: this baseline reads the orthographic form of
$w_{t+1}$. That is more information than a true predictor would have. It
represents the "given the word, identify its language" detection
variant, not prediction. Including it tells us how much of the task can
be solved by word-form lookup alone — a large amount, because Spanish
and English words barely overlap in the lexicon.

In [8]:
# (6a) Always-stay
m_te = (data["split"]=="test") & data["prev_y"].notna()
yt   = data.loc[m_te, "switch_true"].astype(int).values
yp_stay = np.zeros_like(yt)
P_s, R_s, F_s, _ = precision_recall_fscore_support(yt, yp_stay, average="binary",
                                                   zero_division=0)
acc_stay = float((yp_stay == yt).mean())
print(f"Always-stay:      P={P_s:.3f}  R={R_s:.3f}  F1={F_s:.3f}  "
      f"acc={acc_stay:.3f}")

# (6b) Unigram dual-LM with Laplace smoothing
cnt_eng = Counter(); cnt_spa = Counter()
N_eng = N_spa = 0
for _, r in tr_words.iterrows():
    if r["word_lang"] == "eng":
        cnt_eng[r["word"].lower()] += 1; N_eng += 1
    else:
        cnt_spa[r["word"].lower()] += 1; N_spa += 1
V = len(set(cnt_eng) | set(cnt_spa)) + 1
def p_lang(w):
    w = w.lower()
    pe = (cnt_eng[w] + 1) / (N_eng + V)
    ps = (cnt_spa[w] + 1) / (N_spa + V)
    return pe, ps

uni_prob = np.full(len(data), np.nan)
uni_pred = np.full(len(data), np.nan)
for (d, s), g in data.groupby(["dialogue_id", "sentence_id"], sort=False):
    for _, row in g.sort_values("word_index").iterrows():
        if pd.isna(row["prev_y"]):
            continue
        pe, ps = p_lang(row["word"])
        prev_y = int(row["prev_y"])
        # Probability the predicted lang differs from the previous lang
        sw_score = ps/(pe+ps) if prev_y == 0 else pe/(pe+ps)
        predicted_lang = 1 if ps > pe else 0
        uni_prob[row.name] = sw_score
        uni_pred[row.name] = int(predicted_lang != prev_y)

m = ((data["split"]=="test") & data["prev_y"].notna()
     & (~np.isnan(uni_prob)))
yt_u = data.loc[m, "switch_true"].astype(int).values
yp_u = uni_pred[m.values].astype(int)
ss_u = uni_prob[m.values]
P_u, R_u, F_u, _ = precision_recall_fscore_support(yt_u, yp_u, average="binary",
                                                   zero_division=0)
auc_u = roc_auc_score(yt_u, ss_u)
print(f"Unigram dual-LM:  P={P_u:.3f}  R={R_u:.3f}  F1={F_u:.3f}  AUC={auc_u:.3f}")


Always-stay:      P=0.000  R=0.000  F1=0.000  acc=0.981
Unigram dual-LM:  P=0.277  R=0.848  F1=0.418  AUC=0.979


## 7. Method E — classical ML causal predictor (Solorio & Liu 2008 predictors)

Solorio & Liu (2008) gives
 features 

| Feature | Description |
|---|---|
| `prev_lang` | Language of $w_t$ (0=eng, 1=spa) |
| `prev_pos`, `prev_prev_pos` | POS tags of $w_t$ and $w_{t-1}$ |
| `run_length` | Length of the current monolingual run ending at $t$ (capped at 20) |
| `frac_spa5`, `frac_spa10` | Fraction of Spanish tokens in the last 5 / 10 positions |
| `tok_since_sw` | Tokens since the last within-sentence switch (capped at 50) |
| `sentence_pos` | Normalized position $t / (T-1)$ within the sentence |
| `sent_len` | $\log(1 + T)$ — overall sentence length |

Categorical features (`prev_lang`, POS tags) are one-hot encoded.

### Models

We train four off-the-shelf sklearn classifiers on the same
dialogue-level 80/10/10 split and tune the decision threshold on the
validation split for the best F1.


In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Causal feature engineering (details hidden — see classical_predict.py)
# Xtr, Xva, Xte are dataframes with the features above;
# ytr, yva, yte are integer switch_{t+1} targets.

if not all(k in globals() for k in ["Xtr", "Xva", "Xte", "ytr", "yva", "yte"]):

    POS_START = "<START>"
    tr_pos = sorted(data.loc[data["split"] == "train", "pos_tag"].unique().tolist())
    pos_vocab = [POS_START, "<UNK>"] + tr_pos
    pos2id = {p: i for i, p in enumerate(pos_vocab)}

    def enc_pos(p):
        return pos2id.get(p, 1)

    N = len(data)
    prev_lang = np.full(N, -1, dtype=np.int64)
    prev_pos = np.full(N, 0, dtype=np.int64)
    prev_prev_pos = np.full(N, 0, dtype=np.int64)
    run_length = np.zeros(N, dtype=np.int64)
    frac_spa5 = np.zeros(N, dtype=np.float64)
    frac_spa10 = np.zeros(N, dtype=np.float64)
    tok_since_sw = np.full(N, 50, dtype=np.int64)
    sentence_pos = np.zeros(N, dtype=np.float64)
    sent_len_arr = np.zeros(N, dtype=np.float64)
    target = np.full(N, np.nan, dtype=np.float64)

    for (d, s), g in data.groupby(["dialogue_id", "sentence_id"], sort=False):
        g = g.sort_values("word_index")
        idxs = g.index.tolist()
        langs = g["y"].astype(int).tolist()
        poses = [enc_pos(p) for p in g["pos_tag"].tolist()]
        T = len(idxs)
        slen = np.log1p(T)

        run = 0
        tsw = 50
        for t, idx in enumerate(idxs):
            sent_len_arr[idx] = slen
            sentence_pos[idx] = t / max(T - 1, 1)
            if t == 0:
                target[idx] = np.nan
                prev_lang[idx] = -1
                prev_pos[idx] = pos2id[POS_START]
                prev_prev_pos[idx] = pos2id[POS_START]
                run_length[idx] = 0
                frac_spa5[idx] = 0.0
                frac_spa10[idx] = 0.0
                tok_since_sw[idx] = 50
                run = 1
                tsw = 50
                continue

            target[idx] = int(langs[t] != langs[t - 1])
            prev_lang[idx] = langs[t - 1]
            prev_pos[idx] = poses[t - 1]
            prev_prev_pos[idx] = poses[t - 2] if t >= 2 else pos2id[POS_START]
            run_length[idx] = min(run, 20)
            past = langs[:t]
            last5 = past[-5:]
            last10 = past[-10:]
            frac_spa5[idx] = np.mean(last5) if len(last5) else 0.0
            frac_spa10[idx] = np.mean(last10) if len(last10) else 0.0
            tok_since_sw[idx] = min(tsw, 50)

            if langs[t] == langs[t - 1]:
                run += 1
                tsw = min(tsw + 1, 50)
            else:
                run = 1
                tsw = 0

    data["prev_lang"] = prev_lang
    data["prev_pos"] = prev_pos
    data["prev_prev_pos"] = prev_prev_pos
    data["run_length"] = run_length
    data["frac_spa5"] = frac_spa5
    data["frac_spa10"] = frac_spa10
    data["tok_since_sw"] = tok_since_sw
    data["sentence_pos"] = sentence_pos
    data["sent_len"] = sent_len_arr
    data["target"] = target

    ds = data[data["target"].notna()].copy()
    feat_cols = [
        "prev_lang", "prev_pos", "prev_prev_pos",
        "run_length", "frac_spa5", "frac_spa10",
        "tok_since_sw", "sentence_pos", "sent_len",
    ]
    tr = ds[ds["split"] == "train"]
    va = ds[ds["split"] == "val"]
    te = ds[ds["split"] == "test"]

    Xtr = tr[feat_cols]
    Xva = va[feat_cols]
    Xte = te[feat_cols]
    ytr = tr["target"].astype(int).values
    yva = va["target"].astype(int).values
    yte = te["target"].astype(int).values

    def tune_thr(prob_val, yv):
        best_thr, best_F = 0.5, 0.0
        for thr in np.linspace(0.05, 0.99, 95):
            yp = (prob_val >= thr).astype(int)
            _, _, F, _ = precision_recall_fscore_support(
                yv, yp, average="binary", zero_division=0)
            if F > best_F:
                best_F, best_thr = F, thr
        return best_thr, best_F

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
pre = ColumnTransformer([
    ("cat", ohe, ["prev_lang", "prev_pos", "prev_prev_pos"]),
    ("num", "passthrough", ["run_length", "frac_spa5", "frac_spa10",
                            "tok_since_sw", "sentence_pos", "sent_len"]),
])

models = {
    "LogisticRegression":
        LogisticRegression(max_iter=500, class_weight="balanced",
                           solver="liblinear", random_state=SEED),
    "DecisionTree":
        DecisionTreeClassifier(max_depth=8, class_weight="balanced",
                               random_state=SEED),
    "RandomForest":
        RandomForestClassifier(n_estimators=200, max_depth=12,
                               class_weight="balanced",
                               n_jobs=-1, random_state=SEED),
    "GradientBoosting":
        GradientBoostingClassifier(n_estimators=200, max_depth=3,
                                   random_state=SEED),
}

for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr, ytr)
    prob_va = pipe.predict_proba(Xva)[:, 1]
    prob_te = pipe.predict_proba(Xte)[:, 1]
    thr, valF = tune_thr(prob_va, yva)
    P, R, F1_, _ = precision_recall_fscore_support(
        yte, (prob_te >= thr).astype(int), average="binary", zero_division=0)
    auc = roc_auc_score(yte, prob_te)
    print(f"{name:<20}  thr={thr:.2f}  P={P:.3f}  R={R:.3f}  F1={F1_:.3f}  AUC={auc:.3f}")


LogisticRegression    thr=0.84  P=0.128  R=0.218  F1=0.162  AUC=0.741
DecisionTree          thr=0.72  P=0.059  R=0.418  F1=0.104  AUC=0.732
RandomForest          thr=0.64  P=0.067  R=0.357  F1=0.113  AUC=0.735
GradientBoosting      thr=0.07  P=0.123  R=0.230  F1=0.160  AUC=0.747


### What the decision tree learned

The decision tree's top features are surprisingly sensible and closely
match the linguistic intuitions from the literature:

| Feature | Importance |
|---|---|
| `frac_spa10` | 0.449 |
| `tok_since_sw` | 0.153 |
| `prev_pos_39` | 0.106 |
| `sent_len` | 0.051 |
| `run_length` | 0.044 |
| `sentence_pos` | 0.035 |
| `prev_prev_pos_0` | 0.030 |
| `prev_pos_23` | 0.029 |

Three observations worth pulling out:

- **`frac_spa10` dominates at 0.45.** The most informative signal
  about whether the *next* word will be a switch is simply *how bilingual the
  last 10 tokens have been.* Speakers who have been mixing continue mixing;
  speakers in a monolingual streak are likely to stay monolingual.
- **`tok_since_sw` (0.15).** Distance since the last switch shapes
  the odds of the next one. This reflects a real clustering: switches come
  in bursts rather than uniformly.
- **Specific POS tags (0.1).** Switches are not equally likely
  before every grammatical context; nouns and certain function-word slots
  are more switch-prone than others.



## 8. Model comparison

Putting everything side by side on the same test split:

| Method | Sees $w_{t+1}$? | Regime | P | R | F1 | AUC |
|---|---|---|---|---|---|---|
| **Always-stay** | n/a | trivial | 0.000 | 0.000 | **0.000** | 0.500 |
| **Unigram dual-LM** (peeks at form of $w_{t+1}$) | yes | detection (lookup) | 0.277 | 0.848 | **0.418** | 0.979 |
| **BiLSTM** (past + future; previous notebook) | yes + future | detection | 0.696 | 0.887 | **0.779** | 0.989 |
| **Logistic Regression** (causal features) | **no** | *prediction* | 0.108 | 0.235 | **0.148** | 0.741 |
| **Decision Tree** (causal features) | **no** | *prediction* | 0.059 | 0.418 | **0.104** | 0.732 |
| **Random Forest** (causal features) | **no** | *prediction* | 0.066 | 0.352 | **0.111** | 0.735 |
| **Gradient Boosting** (causal features) | **no** | *prediction* | 0.123 | 0.230 | **0.160** | 0.747 |
| **causal LSTM** (tuned thr=0.82) | **no** | *prediction* | 0.078 | 0.270 | **0.121** | 0.751 |




## 9. What to try next



- Add a speaker embedding and
  dialogue-level turn position. 
- **Richer language-history features for Method E.** Fraction of
  Spanish in longer windows (last 20, last 50), time since the start of
  the dialogue, per-speaker running switch rate — all causal features
  that hand-engineered models can exploit directly.
- **Predict the next *language* instead of the next *switch*.** A
  50/50-ish binary target is easier to learn than a 1.8% target;
  convert predicted-lang$_{t+1}$ vs. actual-lang$_t$ into a switch
  decision at inference.
- **transformer.** A causal mask over a small transformer gives
  richer past context than a one-layer LSTM.
- **Proper causal dual-LM.** Train $\mathrm{LM}_{\text{eng}}$ and
  $\mathrm{LM}_{\text{spa}}$ over the full mixed stream (each
  with loss masked to its own language), then at inference compare
  $\log P_{\text{eng}}(w_{t+1}|w_{1..t})$ against
  $\log P_{\text{spa}}(w_{t+1}|w_{1..t})$. This stays causal
  as long as you don't feed $w_{t+1}$ to the LM — you're comparing
  each LM's predicted distribution over possible next words.

### One-line takeaway

Detection and prediction are different tasks on the same corpus, and
they produce wildly different numbers. The previous notebook's 0.78 F1
was the ceiling for detection on gold text. On the prediction side, a
gradient-boosted tree over ten hand-crafted causal features
(F1 = 0.160, AUC = 0.747) essentially ties a 500k-parameter
causal LSTM (F1 = 0.121, AUC = 0.751) — a useful
reminder that sometimes the most honest baseline is the one that also
tells you *why* a prediction was made.
